# 🚀 Getting Started: AgentCore Browser Tool + Browser-Use Integration

## 📖 Overview

This tutorial provides a **comprehensive introduction** to integrating the open-source Browser-Use SDK with Amazon Bedrock AgentCore Browser Tool. You'll learn to build AI-powered web automation solutions that combine enterprise-grade infrastructure with proven browser automation capabilities.

### 🎯 What You'll Learn

- **Headless Browser Automation**: Execute web tasks without visual interfaces
- **AI-Driven Interactions**: Use natural language to control browser actions
- **Enterprise Integration**: Leverage AWS Bedrock's managed infrastructure
- **Session Management**: Handle persistent browser sessions efficiently
- **Error Handling**: Implement robust automation workflows

### 📊 Tutorial Summary

| **Learning Path** | **Duration** | **Complexity** | **Skills Gained** |
|-------------------|--------------|----------------|-------------------|
| **Setup & Configuration** | 10 min | Beginner | Environment setup, SDK installation |
| **Browser Client Integration** | 15 min | Beginner | WebSocket connections, session management |
| **AI-Powered Automation** | 20 min | Intermediate | Natural language task execution |
| **Production Patterns** | 10 min | Intermediate | Error handling, cleanup procedures |

**Total Estimated Time:** 55 minutes

### 🏗️ Tutorial Details

| Information         | Details                                                                                   |
|:--------------------|:-------------------------------------------------------------------------------------------        
| Tutorial type       | Conversational                                                                            |
| Agent type          | Single                                                                                    |
| Agentic Framework   | Browser-Use                                                                               |
| LLM model           | Anthropic Claude 3.7 Sonnet                                                               |
| Tutorial components | Using Browser-Use SDK to interact with Bedrock AgentCore browser tool in a headless way   |
| Tutorial vertical   | Cross-vertical                                                                            |
| Example complexity  | Easy to Intermediate                                                                      |
| SDK used            | Amazon BedrockAgentCore Python SDK, Browser-Use                                           |

### 🏛️ Tutorial Architecture

This tutorial demonstrates the integration of Browser-Use SDK with AgentCore browser tool, creating a powerful hybrid architecture for web automation.

**Architecture Flow:**
1. **Natural Language Input** → AI processes user instructions
2. **Browser-Use Agent** → Translates commands to browser actions
3. **AgentCore Browser Tool** → Executes actions in secure sandbox
4. **Feedback Loop** → Screenshots and results returned to AI

<div style="text-align:left">
    <img src="images/architecture_runtime.png" width="50%"/>
</div>

### ✨ Tutorial Key Features

- **🔒 Secure Execution**: Sandboxed browser environment for safe automation
- **🤖 AI-Powered**: Natural language task execution with Claude 3.7 Sonnet
- **⚡ High Performance**: Optimized for headless browser operations
- **🔄 Session Persistence**: Maintain browser state across multiple tasks
- **📊 Rich Feedback**: Visual progress indicators and detailed error reporting
- **🛠️ Production Ready**: Proper error handling and resource cleanup

### 🎓 Learning Outcomes

By completing this tutorial, you'll master:

#### 🔧 Technical Skills
- **Browser Automation**: Headless browser control and interaction
- **AI Integration**: Natural language to browser action translation
- **Session Management**: Persistent browser sessions and lifecycle management
- **Error Handling**: Robust automation with proper exception management

#### 🏗️ Architecture Understanding
- **Hybrid Integration**: Combining multiple automation frameworks
- **Secure Sandboxing**: Understanding browser tool security models
- **WebSocket Communication**: Real-time browser control protocols
- **Resource Management**: Efficient browser resource utilization

#### 🚀 Practical Applications
- **Web Scraping**: Automated data extraction from websites
- **Testing Automation**: Automated UI testing and validation
- **Process Automation**: Streamlining repetitive web-based tasks
- **Content Management**: Automated content creation and management

---

## Prerequisites

To execute this tutorial you will need:
* Python 3.11+
* AWS credentials
* Amazon Bedrock AgentCore SDK
* Browser-Use SDK 

## How it works

A browser tool sandbox is a secure execution environment that enables AI agents to safely interact with web browsers. When a user makes a request, the Large Language Model (LLM) selects appropriate tools and translates commands. These commands are executed within a controlled sandbox environment containing a headless browser and hosted library server (using tools like Playwright). The sandbox provides isolation and security by containing web interactions within a restricted space, preventing unauthorized system access. The agent receives feedback through screenshots and can perform automated tasks while maintaining system security. 

![architecture local](../images//browser-tool.png)

## 1. Setting Up the Environment

First, let's install and import the necessary libraries to initiaize the browser tool sandbox client. 

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
from bedrock_agentcore.tools.browser_client import BrowserClient
from browser_use import Agent
from browser_use.browser.session import BrowserSession
from browser_use.browser import BrowserProfile
from langchain_aws import ChatBedrockConverse
from rich.console import Console
from contextlib import suppress
import asyncio

In [ ]:
console = Console()

## 2. Setup the browser client
We will setup the browser client and wait for it to be ready. Then, we will generate web-socket url and headers 
 


In [ ]:
from boto3.session import Session
boto_session = Session()
region = boto_session.region_name

client = BrowserClient(region)
client.start()

# Extract ws_url and headers
ws_url, headers = client.generate_ws_headers()

## 4. Helper function to run browser task
Run a browser automation task using browser-use Agent 




In [ ]:
async def run_browser_task(browser_session: BrowserSession, bedrock_chat: ChatBedrockConverse, task: str) -> None:
    """
    Run a browser automation task using browser_use
    
    Args:
        browser_session: Existing browser session to reuse
        bedrock_chat: Bedrock chat model instance
        task: Natural language task for the agent
    """
    try:
        # Show task execution
        console.print(f"\n[bold blue]🤖 Executing task:[/bold blue] {task}")
        
        # Create and run the agent
        agent = Agent(
            task=task,
            llm=bedrock_chat,
            browser_session=browser_session
        )
        
        # Run with progress indicator
        with console.status("[bold green]Running browser automation...[/bold green]", spinner="dots"):
            await agent.run()
        
        console.print("[bold green]✅ Task completed successfully![/bold green]")
        
    except Exception as e:
        console.print(f"[bold red]❌ Error during task execution:[/bold red] {str(e)}")
        import traceback
        if console.is_terminal:
            traceback.print_exc()

## 5. Invoke the browser task using Browser-use profile
Create a persistent browser session using CDP WebSocket connection and initialize Bedrock Claude model for automated web tasks. Handle session lifecycle with proper cleanup and execute browser automation tasks via AI-driven commands.

In [ ]:
# Create persistent browser session and model
browser_session = None
bedrock_chat = None

try:
    # Create browser profile with headers
    browser_profile = BrowserProfile(
        headers=headers,
        timeout=1500000,  # 150 seconds timeout
    )
    
    # Create a browser session with CDP URL and keep_alive=True for persistence
    browser_session = BrowserSession(
        cdp_url=ws_url,
        browser_profile=browser_profile,
        keep_alive=True  # Keep browser alive between tasks
    )
    
    # Initialize the browser session
    console.print("[cyan]🔄 Initializing browser session...[/cyan]")
    await browser_session.start()
    
    # Create ChatBedrockConverse once
    bedrock_chat = ChatBedrockConverse(
        model_id="us.anthropic.claude-3-7-sonnet-20250219-v1:0",
        region_name="us-west-2"
    )
    
    console.print("[green]✅ Browser session initialized and ready for tasks[/green]\n")

    task = "Search for a coffee maker on amazon.com and extract details of the first one" ## Modify the task to run other tasks

    await run_browser_task(browser_session, bedrock_chat, task)

finally:
    # Close the browser session
    if browser_session:
        console.print("\n[yellow]🔌 Closing browser session...[/yellow]")
        with suppress(Exception):
            await browser_session.close()
        console.print("[green]✅ Browser session closed[/green]")


## 6. Cleanup
Stop the browser session if it hasn't been

In [ ]:
client.stop()
print("Browser session stopped successfully!")

---

# 🎉 Tutorial Complete! You've Mastered AgentCore + Browser-Use Integration

## 🏆 Congratulations on Your Achievement!

You've successfully completed the **Getting Started with AgentCore Browser Tool + Browser-Use Integration** tutorial. You now have the foundational skills to build AI-powered web automation solutions using enterprise-grade infrastructure.

### 📋 What You've Accomplished

#### ✅ Core Skills Mastered
- **Environment Setup**: Configured Python environment with AgentCore and Browser-Use SDKs
- **Browser Client Integration**: Established WebSocket connections to AgentCore browser tool
- **AI-Powered Automation**: Created agents that understand natural language instructions
- **Session Management**: Implemented persistent browser sessions with proper lifecycle management
- **Error Handling**: Built robust automation with comprehensive exception handling
- **Resource Cleanup**: Ensured proper resource management and session termination

#### 🏗️ Architecture Understanding
- **Hybrid Integration**: Successfully combined Browser-Use SDK with AgentCore infrastructure
- **Secure Sandboxing**: Leveraged AWS Bedrock's secure browser execution environment
- **WebSocket Communication**: Established real-time communication with browser instances
- **AI Model Integration**: Connected Claude 3.7 Sonnet for intelligent task execution

### 🚀 Next Steps & Advanced Learning

#### 🔄 Immediate Actions
1. **Experiment with Tasks**: Try different automation scenarios with your setup
2. **Explore Parameters**: Adjust timeout settings and browser profiles for optimization
3. **Test Edge Cases**: Handle different websites and complex interactions
4. **Monitor Performance**: Observe resource usage and execution times

#### 📈 Advanced Tutorials
Ready to take your skills to the next level? Explore these advanced tutorials:

- **🎥 Live View Integration**: `agentcore-browser-tool-live-view-with-browser-use.ipynb`
  - Real-time browser visualization and monitoring
  - Interactive debugging and development workflows
  - Advanced session management techniques

- **🛡️ Enterprise CAPTCHA Handling**: `captcha-handling/agentcore-enterprise-captcha-handling.ipynb`
  - Production-grade CAPTCHA solving with AI
  - Advanced error handling and retry mechanisms
  - Enterprise security and compliance patterns

#### 🏢 Production Implementation
- **Scaling Strategies**: Implement multi-session browser automation
- **Monitoring & Logging**: Add comprehensive observability to your automations
- **Security Hardening**: Implement enterprise security best practices
- **Performance Optimization**: Fine-tune for high-volume automation scenarios

### 🎯 Key Takeaways

#### 💡 Best Practices Learned
1. **Session Persistence**: Reusing browser sessions improves performance and reduces overhead
2. **Proper Cleanup**: Always ensure browser sessions are properly closed to prevent resource leaks
3. **Error Handling**: Comprehensive exception handling is crucial for production automation
4. **Natural Language**: AI agents can understand complex, natural language instructions
5. **Secure Execution**: AgentCore provides enterprise-grade security for browser automation

#### 🔮 Future Opportunities
- **Multi-Modal AI**: Combine text and vision AI for advanced web interactions
- **Workflow Orchestration**: Build complex, multi-step automation workflows
- **Custom Integrations**: Extend the framework for specific business use cases
- **Performance Analytics**: Implement detailed performance monitoring and optimization

### 📚 Additional Resources

#### 🔗 Related Tutorials
- **NovaAct Integration**: Explore alternative browser automation frameworks
- **Advanced Browser Patterns**: Learn complex interaction techniques
- **Production Deployment**: Scale your automations to enterprise environments

#### 📖 Documentation & Support
- **AgentCore Documentation**: Deep dive into browser tool capabilities
- **Browser-Use SDK**: Explore advanced features and customization options
- **AWS Bedrock**: Understand the underlying AI and infrastructure services

### 🤝 Community & Sharing

#### 💬 Getting Help
- **Common Issues**: Review error handling patterns for troubleshooting
- **Performance Questions**: Optimize browser session configurations
- **Integration Challenges**: Explore advanced integration patterns

#### 🌟 Sharing Success
- **Document Patterns**: Create reusable automation templates
- **Share Learnings**: Help others master these integration techniques
- **Contribute Examples**: Expand the community knowledge base

---

## 🎊 Ready for Advanced Automation!

You've built a solid foundation in AI-powered browser automation. The skills you've developed will serve you well in creating sophisticated, enterprise-ready web automation solutions. Keep exploring, keep building, and keep pushing the boundaries of what's possible with AI and browser automation!

**Happy Automating! 🤖✨**
